In [26]:
# import packages
import csv
import numpy as np
import pandas as pd

In [29]:
#Daftar file dan metadata
files = [
    ("sd_2021.csv", "SD", 2021),
    ("sd_2022.csv", "SD", 2022),
    ("sd_2023.csv", "SD", 2023),
    ("sd_2024.csv", "SD", 2024),
]

In [30]:
# List untuk menyimpan data
df_all = []

In [31]:
# Loop untuk membaca dan memproses tiap file
for file, jenjang, tahun in files:
    try:
        # Buat path lengkap
        file_path = f"/content/sample_data/dataset_sd/{file}"

        # Baca file CSV
        df = pd.read_csv(file_path)

        # Normalisasi nama kolom
        df = df.rename(columns={
            "Laki-laki (L) - Negeri": "L_Negeri",
            "Laki-laki (L) - Swasta": "L_Swasta",
            "Laki-laki (L) - Subjml": "L_Jumlah",
            "Perempuan (P) - Negeri": "P_Negeri",
            "Perempuan (P) - Swasta": "P_Swasta",
            "Perempuan (P) - Subjml": "P_Jumlah",
        })

        # Tambah metadata Jenjang dan Tahun
        df["Jenjang"] = jenjang
        df["Tahun"] = tahun

        # Ubah ke format long
        df_long = pd.melt(
            df,
            id_vars=["Kecamatan", "Jenjang", "Tahun"],
            value_vars=["L_Negeri", "L_Swasta", "L_Jumlah", "P_Negeri", "P_Swasta", "P_Jumlah"],
            var_name="Kategori",
            value_name="Jumlah"
        )

        # Tambahkan kolom Jenis Kelamin dan Tipe Sekolah
        df_long["Jenis Kelamin"] = df_long["Kategori"].apply(lambda x: "Laki-laki" if x.startswith("L") else "Perempuan")
        df_long["Tipe Sekolah"] = df_long["Kategori"].apply(lambda x: "Negeri" if "Negeri" in x else ("Swasta" if "Swasta" in x else "Total"))

        # Hapus kolom Kategori
        df_long = df_long.drop(columns="Kategori")

        # Simpan dataframe sementara ke list
        df_all.append(df_long)

    except Exception as e:
        print(f"Gagal memproses file {file}: {e}")

In [32]:
# Gabungkan semua jadi satu DataFrame besar
df_final = pd.concat(df_all, ignore_index=True)

In [33]:
# Tampilkan hasil
print(df_final.head())

                   Kecamatan Jenjang  Tahun  Jumlah Jenis Kelamin Tipe Sekolah
0    Kec. Teluk Betung Barat      SD   2021       0     Laki-laki       Negeri
1  Kec. Teluk Betung Selatan      SD   2021       0     Laki-laki       Negeri
2               Kec. Panjang      SD   2021       0     Laki-laki       Negeri
3  Kec. Tanjung Karang Timur      SD   2021       2     Laki-laki       Negeri
4    Kec. Teluk Betung Utara      SD   2021       2     Laki-laki       Negeri


In [34]:
# Simpan ke CSV
df_final.to_csv("dataset_sd.csv", index=False)